# Persona Multi-LoRA workshop: select one client and prepare replies

**Primary question:** How do we turn one reviewed persona-response CSV into leakage-safe train, validation, and sealed test partitions for a client-specific LoRA adapter?

**Time:** 20 minutes. This notebook performs validation and preparation only; it does not load a language model.

Prerequisite: review the completed response datasets and persona definitions under `data/`.


## Data contract

- Select exactly one client with `CLIENT_ID` at the top of the configuration cell.
- `INPUT_FILE` is resolved from `config/persona_registry.json`; the registry prevents pairing one client's replies with another client's profile.
- Preserve the existing train, validation, and test assignments from the reviewed CSV.
- Train only on `train`, select checkpoints only with `validation`, and keep `test` sealed until notebook 06.
- Treat `personality_response` as the supervised completion. Labels remain provenance only; they are not the response target.
- Do not print subject/body text during data inspection.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

import email_workshop
from email_workshop.data import write_json
from email_workshop.persona import load_persona_dataset, load_persona_registry, persona_messages

WORKSHOP_ROOT = Path(email_workshop.__file__).resolve().parents[1]
PERSONA_REGISTRY = load_persona_registry(WORKSHOP_ROOT)
# Change only CLIENT_ID to select another registered client.
CLIENT_ID = 'maya-chen-ceo'
INPUT_FILE = WORKSHOP_ROOT / PERSONA_REGISTRY[CLIENT_ID]['data_file']
PERSONA_RUN_ROOT = WORKSHOP_ROOT / Path(os.getenv('EMAIL_PERSONA_RUN', 'artifacts/persona-lora-run-2026'))
SHOW_PROFILE_PROMPT = False

print(f'Kernel: {sys.executable}')
print(f'Client: {CLIENT_ID}')
print(f'Input file: {INPUT_FILE}')
print(f'Persona artifacts: {PERSONA_RUN_ROOT}')


h:\masterswork\gitrash\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Kernel: h:\masterswork\gitrash\.venv\Scripts\python.exe
Client: maya-chen-ceo
Input file: H:\masterswork\gitrash\multi-lora\workshops\email-multilora\data\manual_full_response\01-maya-chen-ceo-email-responses.csv
Persona artifacts: H:\masterswork\gitrash\multi-lora\workshops\email-multilora\artifacts\persona-lora-run-2026


## Inspect the client registry

The registry is the single mapping from a client ID to the person's display name, role, adapter name, profile Markdown, and reviewed response CSV. A training run never guesses these relationships from filenames.

To train another client later, change `CLIENT_ID` and rerun all three persona notebooks with the same shared `PERSONA_RUN_ROOT`. Each client writes to a separate adapter directory.


In [2]:
registry_table = pd.DataFrame.from_dict(PERSONA_REGISTRY, orient='index')
registry_table.index.name = 'client_id'
display(registry_table)
if CLIENT_ID not in PERSONA_REGISTRY:
    raise ValueError(f'Choose one of {tuple(PERSONA_REGISTRY)}')


,name,role,adapter,profile_file,data_file
client_id,,,,,
maya-chen-ceo,Maya Chen,Chief Executive Officer,maya-chen,data/personalities/01-maya-chen-ceo.md,data/manual_full_response/01-maya-chen-ceo-ema...
daniel-okafor-cfo,Daniel Okafor,Chief Financial Officer,daniel-okafor,data/personalities/02-daniel-okafor-cfo.md,data/manual_full_response/02-daniel-okafor-cfo...
priya-raman-support,Priya Raman,L1 Customer Support Specialist,priya-raman,data/personalities/03-priya-raman-l1-support-s...,data/manual_full_response/03-priya-raman-l1-su...
lucas-silva-engineer,Lucas Silva,Software Engineer,lucas-silva,data/personalities/04-lucas-silva-software-eng...,data/manual_full_response/04-lucas-silva-softw...
amina-yusuf-sales,Amina Yusuf,Sales Account Executive,amina-yusuf,data/personalities/05-amina-yusuf-sales-accoun...,data/manual_full_response/05-amina-yusuf-sales...
jordan-lee-people-ops,Jordan Lee,People Operations Manager,jordan-lee,data/personalities/06-jordan-lee-people-operat...,data/manual_full_response/06-jordan-lee-people...
elena-petrova-marketing,Elena Petrova,Marketing Director,elena-petrova,data/personalities/07-elena-petrova-marketing-...,data/manual_full_response/07-elena-petrova-mar...
marcus-green-legal,Marcus Green,Legal and Compliance Counsel,marcus-green,data/personalities/08-marcus-green-legal-compl...,data/manual_full_response/08-marcus-green-lega...
sofia-alvarez-operations,Sofia Alvarez,Business Operations Manager,sofia-alvarez,data/personalities/09-sofia-alvarez-operations...,data/manual_full_response/09-sofia-alvarez-ope...


## Load and verify the selected client

The loader checks the committed manifest checksum, exact schema, unique IDs, non-empty replies, profile presence, and all three split names. An explicit `INPUT_FILE` must match the selected registry entry, preventing silent profile/data swaps.


In [3]:
splits, persona, profile_text, data_manifest = load_persona_dataset(
    WORKSHOP_ROOT,
    CLIENT_ID,
    INPUT_FILE,
)
display(pd.Series(persona, name='selected_persona'))
display(pd.Series(data_manifest['split_counts'], name='rows'))
print(f"Dataset SHA-256: {data_manifest['dataset_sha256']}")
print(f"Profile SHA-256: {data_manifest['profile_sha256']}")


name                                                    Maya Chen
role                                      Chief Executive Officer
adapter                                                 maya-chen
profile_file               data/personalities/01-maya-chen-ceo.md
data_file       data/manual_full_response/01-maya-chen-ceo-ema...
Name: selected_persona, dtype: object

train         1473
validation     312
test           318
Name: rows, dtype: int64

Dataset SHA-256: 576d403752315f569b83071be06ba881508e16165f360ee50d03acfd97287e6d
Profile SHA-256: 7b13641be3bde3ee1e10af2af5dfb013c186c06c16c42882ce7a35c3bf649e3f


## Inspect response targets without exposing emails

Response length is a useful capacity check. Very long replies need a larger sequence budget; unusually short replies may indicate a weak training target. The code reports only aggregate lengths and label counts, not email contents.


In [4]:
summary_rows = []
for split_name, rows in splits.items():
    word_counts = [len(row['personality_response'].split()) for row in rows]
    summary_rows.append({
        'split': split_name,
        'rows': len(rows),
        'response_words_min': min(word_counts),
        'response_words_median': float(pd.Series(word_counts).median()),
        'response_words_p95': float(pd.Series(word_counts).quantile(0.95)),
        'response_words_max': max(word_counts),
        'multi_label_emails': sum(row['label_count'] > 1 for row in rows),
    })
display(pd.DataFrame(summary_rows).set_index('split'))


,rows,response_words_min,response_words_median,response_words_p95,response_words_max,multi_label_emails
split,,,,,,
train,1473,5,22.0,29.0,36,783
validation,312,9,19.0,29.0,33,170
test,318,6,19.0,27.0,31,167


## Build the response-writer prompt

Every adapter sees the same prompt shape:

1. A system message identifies the selected person and includes the reviewed synthetic profile.
2. A user message contains the incoming subject and body.
3. The supervised completion is the authored `personality_response` only.

The profile remains in the prompt at inference time. This makes the base-versus-adapter comparison fair: both receive the same personality instructions, while only the adapted model receives learned LoRA weights.


In [6]:
profile_text

"# Maya Chen — Chief Executive Officer\n\n> Fictional workshop persona. This profile is synthetic and must not be treated as a real\n> employee record or inferred from private email.\n\n## Organizational context\n\nMaya leads company strategy, board communication, executive alignment, major partnerships,\nand high-impact organizational decisions. Her inbox receives broad context but she personally\nacts on a small number of decisions.\n\n## Personality traits\n\n| Trait | Expression at work |\n|---|---|\n| Strategic | Connects an email to long-term company direction before acting |\n| Decisive | Prefers a clear recommendation, owner, deadline, and requested decision |\n| Curious | Asks what evidence would change the proposed direction |\n| Time-conscious | Skims quickly and delegates operational detail |\n| Calm under pressure | Avoids escalating tone until facts and consequences are clear |\n| Relationship-oriented | Pays attention to stakeholder trust and external commitments |\n\n##

In [7]:
synthetic_messages = persona_messages(
    persona,
    profile_text,
    'Quarterly planning follow-up',
    'Please review the proposal and reply with the next decision and owner.',
)
if SHOW_PROFILE_PROMPT:
    display(pd.DataFrame(synthetic_messages))
else:
    display(pd.DataFrame([
        {'role': item['role'], 'characters': len(item['content']), 'preview': item['content'][:1600] + '...'}
        for item in synthetic_messages
    ]))


,role,characters,preview
0,system,2831,"You write email replies as Maya Chen, Chief Ex..."
1,user,168,SUBJECT\nQuarterly planning follow-up\n\nEMAIL...


In [8]:
print('SYSTEM PROMPT')
print('=' * 80)
print(synthetic_messages[0]['content'])
print('\nUSER PROMPT')
print('=' * 80)
print(synthetic_messages[1]['content'])


SYSTEM PROMPT
You write email replies as Maya Chen, Chief Executive Officer. Follow the synthetic workshop profile below. Preserve its communication style and boundaries without mentioning the profile or claiming actions that the email does not support. Return only the reply body, with no subject line, analysis, or wrapper.

PERSONA PROFILE
# Maya Chen — Chief Executive Officer

> Fictional workshop persona. This profile is synthetic and must not be treated as a real
> employee record or inferred from private email.

## Organizational context

Maya leads company strategy, board communication, executive alignment, major partnerships,
and high-impact organizational decisions. Her inbox receives broad context but she personally
acts on a small number of decisions.

## Personality traits

| Trait | Expression at work |
|---|---|
| Strategic | Connects an email to long-term company direction before acting |
| Decisive | Prefers a clear recommendation, owner, deadline, and requested decision

## Persist the selected data identity

Notebook 05 reloads the committed CSV and profile rather than trusting notebook memory. This small handoff manifest records the exact selection for human review; the training manifest will independently include the same hashes.


In [9]:
handoff = {
    **data_manifest,
    'input_file': INPUT_FILE.relative_to(WORKSHOP_ROOT).as_posix(),
    'persona_run_root': str(PERSONA_RUN_ROOT),
}
write_json(PERSONA_RUN_ROOT / 'persona_data' / f"{persona['adapter']}.json", handoff)
display(pd.Series(handoff))


client_id                                               maya-chen-ceo
persona             {'name': 'Maya Chen', 'role': 'Chief Executive...
data_file           data/manual_full_response/01-maya-chen-ceo-ema...
profile_file                   data/personalities/01-maya-chen-ceo.md
dataset_sha256      576d403752315f569b83071be06ba881508e16165f360e...
profile_sha256      7b13641be3bde3ee1e10af2af5dfb013c186c06c16c428...
rows                                                             2103
split_counts          {'train': 1473, 'validation': 312, 'test': 318}
split_ids_sha256    {'train': '25c738e05fbaa9865b85e1f73a77c3ec004...
task                    persona-conditioned email response generation
test_policy                             sealed until final evaluation
input_file          data/manual_full_response/01-maya-chen-ceo-ema...
persona_run_root    H:\masterswork\gitrash\multi-lora\workshops\em...
dtype: object

## Ready for notebook 05 when

- The selected client, profile, and input CSV match.
- Train, validation, and test are all non-empty.
- Response lengths fit the planned context window.
- The test partition has not been inspected or used for tuning.
